In [1]:
import os
from dotenv import load_dotenv 

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


if os.getenv("GROQ_API_KEY"):
    print("I can read all")



I can read all


In [2]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

model = ChatGroq(model="llama-3.1-8b-instant")
model.invoke("What is todays date").content

"Today's date is July 1, 2026."

##### ***TOOLS***


In [3]:
# TOOL - 1 [News tool search]

from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(description="Use to search the web")

search_tool.invoke("what is my name")

C:\Users\HP\AppData\Local\Temp\ipykernel_22544\2934304360.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


'Names.org is a website that helps you discover the meaning, origin, popularity, and trends of your name. You can search by first name, last name, full name, or baby name, and explore various lists and categories of names. WhatsMyName is a search engine that finds where a specific username exists across the internet. Instead of searching for web pages, it searches for you. May 23, 2026 · Ever wondered about the history and meaning behind your first name? Enter it here to uncover its cultural origins, hidden meanings, and fascinating facts. Feb 23, 2026 · Answer these questions about yourself, and we’ll guess your first name with stunning (ish) accuracy. Curious to see if we can pull it off? Click that “Start Quiz” button to find out! Take a fun quiz based on your personality traits and see if we can accurately guess your name. From symbolic meanings to cultural significance, names play a huge role in shaping who we are.'

In [6]:
# TOOL - 2 [Wikipedia search tool]

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=4000,
)

wikipedia_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(wikipedia_tool.invoke("Generative Artificial Intelligence"))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# TOOL-3 
from langchain.tools import tool

@tool
def enterprise_tool(query:str)-> str:
    """Its used to send email to client"""
    return "Email Sent"

In [ ]:
Toolkit = [search_tool, wikipedia_tool, enterprise_tool]

In [ ]:
Toolkit

[DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text')),
 WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\HP\\Desktop\\Agentic walkflow tutorial\\langraph tutorial\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 StructuredTool(name='enterprise_tool', description='Its used to send email to client', args_schema=<class 'langchain_core.utils.pydantic.enterprise_tool'>, func=<function enterprise_tool at 0x00000172E3A77560>)]

### ***ReAct Agent***

In [9]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1000,
    timeout=30,
)

agent = create_agent(model, tools=Toolkit)
agent

NameError: name 'Toolkit' is not defined

#### ***ReAct agent streaming***

In [ ]:
example_query = "Give me the latest news about the stock market"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Give me the latest news about the stock market
================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (7czcyqs3f)
 Call ID: 7czcyqs3f
  Args:
    query: latest stock market news
================================= Tool Message =================================
Name: duckduckgo_search

MarketWatch provides the latest stock market, financial and business news. Get stock market quotes, personal finance advice, company news and more. At Yahoo Finance, you get free stock quotes, up-to-date news, portfolio management resources, international market data, social interaction and mortgage rates that help you manage your financial life. News and analysis of markets across the globe.Explore this week’s top tech news and market movers, plus key catalysts to watch next week. 26 June. Technology Investing. Latest Articles. Top 5 Canadian Mining Stocks This Week: G

In [8]:
import requests
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_groq import ChatGroq


@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia for a topic and return a summary."""

    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + query.replace(" ", "_")

    headers = {
        "User-Agent": "MyLangChainApp/1.0 (contact@example.com)"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        data = response.json()

        if "extract" in data:
            return data["extract"]

        return "No summary found."

    except requests.exceptions.RequestException as e:
        return f"Request failed: {e}"
    except ValueError:
        return "Wikipedia returned an invalid response."


model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1000,
)

agent = create_agent(
    model=model,
    tools=[wikipedia_search],
)

query = "Tell me about Generative Artificial Intelligence"

for event in agent.stream(
    {"messages": [("user", query)]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about Generative Artificial Intelligence
================================== Ai Message ==================================
Tool Calls:
  wikipedia_search (ym43nnb7n)
 Call ID: ym43nnb7n
  Args:
    query: Generative Artificial Intelligence
================================= Tool Message =================================
Name: wikipedia_search

Request failed: 404 Client Error: Not Found for url: https://en.wikipedia.org/api/rest_v1/page/summary/Generative_Artificial_Intelligence
================================== Ai Message ==================================

Generative Artificial Intelligence (AI) refers to a type of artificial intelligence that is capable of generating new, original content, such as images, videos, music, text, and more. This is in contrast to traditional AI, which is typically focused on analyzing and processing existing data.

Generative AI uses complex algorithms and neural net